# F1-scientific-python — Practice p23

**Type:** constrained coding · **Difficulty:** core · **Concepts:** seaborn-programming

**Set B — exam register · Budget: 30 minutes.**

Implement exactly `def plot_cohort_scatter(hours, scores, cohorts):`. All three inputs are 1-D NumPy arrays of equal length; `hours` and `scores` are numeric, while `cohorts` contains the strings `morning` and `evening`.

Your function must:

- create `fig, ax = plt.subplots(figsize=(6, 4))`;
- call `sns.scatterplot` exactly once with `x=hours`, `y=scores`, `hue=cohorts`, `style=cohorts`, `s=80`, and `ax=ax`;
- set exact title `Practice time and quiz score`, x-label `practice hours`, and y-label `quiz score`;
- keep the automatically generated legend and return `(fig, ax)`;
- not call `plt.show()` inside the function.

In the markdown response after the check, explain what information `hue` and `style` each encode and why fixed `color=...` would not satisfy the group-semantic contract.

**Banned (zero points): `plt.scatter` inside `plot_cohort_scatter`, manually drawing one call per cohort, and any non-array input conversion.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import to_rgba

In [ ]:
def plot_cohort_scatter(hours, scores, cohorts):
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.scatterplot(
        x=hours,
        y=scores,
        hue=cohorts,
        style=cohorts,
        s=80,
        ax=ax,
    )
    ax.set_title("Practice time and quiz score")
    ax.set_xlabel("practice hours")
    ax.set_ylabel("quiz score")
    return fig, ax

Run the deterministic contract check below. Do not edit the check.

In [ ]:
probe_hours = np.array([1.0, 1.5, 2.0, 2.5, 3.0, 3.5])
probe_scores = np.array([51.0, 63.0, 65.0, 74.0, 72.0, 86.0])
probe_cohorts = np.array(["morning", "evening", "morning", "evening", "morning", "evening"])
scatterplot_calls = []
_original_scatterplot = sns.scatterplot
_original_show = plt.show

def _record_scatterplot(*args, **kwargs):
    scatterplot_calls.append((args, kwargs.copy()))
    return _original_scatterplot(*args, **kwargs)

def _forbid_show(*args, **kwargs):
    raise AssertionError("plot_cohort_scatter must not call plt.show()")

sns.scatterplot = _record_scatterplot
plt.show = _forbid_show
try:
    fig, ax = plot_cohort_scatter(probe_hours, probe_scores, probe_cohorts)
finally:
    sns.scatterplot = _original_scatterplot
    plt.show = _original_show

legend_text = [item.get_text() for item in ax.get_legend().get_texts()]
data_collections = [collection for collection in ax.collections if collection.get_offsets().shape[0] > 0]
plotted_offsets = np.vstack([collection.get_offsets() for collection in data_collections])
legend = ax.get_legend()
legend_handles = legend.legend_handles

def _effective_alpha(artist):
    explicit_alpha = artist.get_alpha()
    if explicit_alpha is not None:
        return float(explicit_alpha)
    color_alphas = []
    for accessor in ("get_facecolor", "get_edgecolor"):
        if hasattr(artist, accessor):
            colors = np.asarray(getattr(artist, accessor)())
            if colors.size:
                color_alphas.extend(np.atleast_2d(colors)[:, -1].tolist())
    if hasattr(artist, "get_color"):
        color_alphas.append(to_rgba(artist.get_color())[3])
    return max(color_alphas, default=1.0)

assert len(scatterplot_calls) == 1
call_args, call_kwargs = scatterplot_calls[0]
assert call_args == ()
required_kwargs = {"x", "y", "hue", "style", "s", "ax"}
assert required_kwargs <= set(call_kwargs)
assert call_kwargs["x"] is probe_hours
assert call_kwargs["y"] is probe_scores
assert call_kwargs["hue"] is probe_cohorts
assert call_kwargs["style"] is probe_cohorts
assert call_kwargs["s"] == 80 and call_kwargs["ax"] is ax
assert len(data_collections) == 1
assert data_collections[0].get_visible()
assert _effective_alpha(data_collections[0]) > 0
assert np.allclose(plotted_offsets, np.column_stack([probe_hours, probe_scores]), atol=1e-12, rtol=0)
assert np.allclose(data_collections[0].get_sizes(), 80, atol=1e-12, rtol=0)
assert set(legend_text) == {"morning", "evening"}
assert legend.get_visible() and len(legend_handles) == 2
assert all(handle.get_visible() and _effective_alpha(handle) > 0 for handle in legend_handles)
assert all(item.get_visible() and _effective_alpha(item) > 0 for item in legend.get_texts())
assert np.allclose(fig.get_size_inches(), (6, 4), atol=1e-12, rtol=0)
assert ax.get_title() == "Practice time and quiz score"
assert ax.get_xlabel() == "practice hours" and ax.get_ylabel() == "quiz score"
plt.close(fig)

### Semantic explanation

*Write your explanation here.*

`hue=cohorts` maps cohort membership to color, while `style=cohorts` maps the same membership to marker shape. These redundant channels keep the morning and evening groups distinguishable when color alone is difficult to perceive. A fixed `color=...` would give every point one color, so it would encode no cohort semantics and could not produce the required two-group legend mapping.

### Answer check

The immutable assertions verify one keyword-only `sns.scatterplot` call with the original arrays, one visible data collection containing every ordered point, marker size 80, a visible two-entry semantic legend, figure size, title, and labels. The function never calls `plt.show()`; the check temporarily replaces it with a raising sentinel.